In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.func import vmap, grad, functional_call, jacrev, vjp
import wandb
from accelerate.test_utils.testing import get_backend
from typing import Optional
from core.models import NoisyMLP, WandBCallback, load_model_from_artifact
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
from core.data import MNISTDataModule
from lightning.pytorch.callbacks import EarlyStopping
device, n_devices, _ = get_backend()

torch.set_float32_matmul_precision("high") 

## Fitting gradients of a known function class

In [ ]:
# Define your RFunction model (unchanged)
class RFunction(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.tensor([1.0, 1.0]))  # Initialize parameters

    def forward(self, A):
        return torch.sum(self.theta[0] * torch.exp(self.theta[1] * A))  # Example function R(A, θ)

# For this example, we'll use the same simulated data, but in batches
A_data = torch.randn(500, 5)  # More data points
true_theta = torch.tensor([2.0, 0.5]) 
true_gradients = true_theta[0] * true_theta[1] * torch.exp(true_theta[1] * A_data)  # Simulated ∇R(A_i)
# Make sure to reshape true_gradients to match the batch size

# Create dataset and dataloader
dataset = TensorDataset(A_data, true_gradients)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Instantiate model
model = RFunction()
optimizer = optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# Training loop with batches
num_epochs = 30

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for A_batch, true_gradients_batch in dataloader:
        optimizer.zero_grad()
        
        # Make inputs require gradients
        A_batch = A_batch.detach().requires_grad_()
        
        # Compute predicted gradient ∇R(A, θ) via autograd
        R_values = model(A_batch)
        gradients = torch.autograd.grad(R_values, A_batch, create_graph=True)[0]
        
        # Compute loss for this batch

        loss = loss_fn(gradients, true_gradients_batch)
        
        # Backpropagate and update θ
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1
    
    avg_epoch_loss = epoch_loss / batch_count
    print(f"Epoch {epoch}: Average Loss = {avg_epoch_loss:.8f}")

# Learned parameters
print("Learned theta:", model.theta.detach().numpy())

## Linear Network with L1 / L2 penalty

In [2]:
from core.models import WandBCallback, LinearNetwork
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
    
# Instantiate the model
input_dim = 10
output_dim = 1
hidden_dim = 200

model = LinearNetwork(input_dim, output_dim, hidden_dim, 
                      l1_lambda=1., l1_smooth=0.01,
                      l2_lambda=0.0, 
                      lr=1e-3)

# fake data
N = 5000
X = torch.randn(N, input_dim)
betas = 5*torch.randn(input_dim) 
y = (betas @ X.T).view(-1, 1) + 0.5 * torch.randn(N, 1)  # Linear relationship with noise
# Create a DataLoader for training and testing
dataset = TensorDataset(X, y)
# train test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=3, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=3, drop_last=True)


logger = WandbLogger(project="inductive-bias", name="linear-net")
trainer = Trainer(max_epochs=200, 
                  logger=logger, 
                  callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss", patience=15, mode="min")], 
                  accelerator=device, 
                  devices=n_devices)
trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=test_dataloader)


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | linear    | Sequential | 2.4 K  | train
1 | loss_func | MSELoss    | 0      | train
-------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

epoch,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/loss,█▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇███
val/loss,█▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,110
train/loss,12.40884
trainer/global_step,13874
val/loss,12.34381


### Using per-sample gradients to estimate bias

In [ ]:
from core.bias import RidgeBias, SmoothLassoBias, LassoBias, ElasticNet

# For this example, we'll use the same simulated data, but in batches
params = dict(model.named_parameters())
flattened_params = torch.cat([p.view(-1) for p in params.values()])
# buffers = dict(model.named_buffers())

# Define function that returns model output
def model_output(params, x):
    return functional_call(model, params, (x,))

# Create dataset and dataloader
dataset = TensorDataset(X, y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Instantiate model
# r_model = RidgeBias()
# wandb.init(project="inductive-bias", name="ridge-bias")
# r_model = SmoothLassoBias(smooth=0.1, alpha_init=0.5)
# wandb.init(project="inductive-bias", name="smooth-lasso-bias")
# r_model = LassoBias()
# wandb.init(project="inductive-bias", name="nonsmooth-lasso-bias")
r_model = ElasticNet()
wandb.init(project="inductive-bias", name="elastic-net")

wandb.watch(r_model)
optimizer = optim.Adam(r_model.parameters(), lr=1e-3)

# Training loop with batches
num_epochs = 200

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for X_batch, y_batch in dataloader:
        # Zero the gradients
        optimizer.zero_grad()
        flattened_params = flattened_params.detach().requires_grad_()

        # vector-Jacobian product, returns function (model_output) applied to primals (params, X) 
        # and a function that computes the vector-Jacobian product
        predictions, vjp_func = vjp(model_output, params, X_batch)
        # vjp with residuals, then select the derivative w.r.t. params
        # factor of 2 comes from gradient of the mse loss
        vjp_result = vjp_func(2*(y_batch - predictions))[0] # is this sign convention correct?
        true_gradients_batch = torch.cat([v.view(-1) for v in vjp_result.values()]) / batch_size

        # Compute predicted gradient ∇R(A, θ) via autograd
        R_val = r_model(flattened_params)
        gradients = torch.autograd.grad(R_val, flattened_params, create_graph=True)[0]
        # Compute loss for this batch
        loss = torch.nn.functional.mse_loss(
            gradients, 
            true_gradients_batch, 
            reduction='mean')
        
        # Backpropagate and update θ
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1

        ## Early stopping

    
    avg_epoch_loss = epoch_loss / batch_count
    # print(f"Epoch {epoch}: Average Loss = {avg_epoch_loss:.8f}")
    wandb.log({"loss": avg_epoch_loss})
    # wandb.log({"learning_rate": optimizer.param_groups[0]['lr']})
    # Learned parameters
    for name, param in r_model.named_parameters():
        # print(f"Parameter {name}: {param.detach().numpy()}")
        wandb.log({name: param.detach().numpy()})

wandb.finish()

lambda_1,▁▂▆▇▇████████▇█▇███▇▇██▇███████▇██████▇█
lambda_2,█▆▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,█▅▆▇▆▆▁▃▆▃▂▅▃▃▅▂▃▅▃▃▄▂▁▂▅▅▁▃▄▃▅▂▅▅▅▃▅▆▁▄
lambda_1,0.97872
lambda_2,0.00466
loss,0.0062


In [ ]:
from core.bias import BiasWithMSE, BiasWithAutodiffLoss, ElasticNet
dataset = TensorDataset(X, y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
bias_estimator = BiasWithMSE(predictive_model=model,
                                      predictive_loss_fn=torch.nn.functional.mse_loss, 
                                      bias_model=ElasticNet(smooth=.01), 
                                      grad_match_loss_fn=torch.nn.functional.mse_loss,)
bias_trainer = Trainer(
    max_epochs=200,
    accelerator=device,
    devices=n_devices,
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss", patience=15, mode="min")],
    logger=WandbLogger(project="inductive-bias", name="linear-ridge")
)
bias_trainer.fit(bias_estimator, train_dataloaders=dataloader)

/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



  | Name             | Type          | Params | Mode 
-----------------------------------------------------------
0 | predictive_model | LinearNetwork | 2.4 K  | train
1 | bias_model       | ElasticNet    | 2      | train
-----------------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/lambda_1,█▇▅▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
bias/lambda_2,█▇▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/loss,█▇▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
bias/lambda_1,0.0303
bias/lambda_2,-2e-05
epoch,44
train/loss,1e-05
trainer/global_step,6999


#### Testing vjp

In [ ]:
params = dict(model.named_parameters())
# buffers = dict(model.named_buffers())

# Define function that returns model output
def model_output(params, x):
    return functional_call(model, params, (x,))

# def single_output(params, x_single):
#     # Return a scalar prediction per example 
#     return functional_call(model, params, (x_single,)).squeeze()

# Compute Jacobian of output w.r.t. weights
# x = torch.randn(10, input_dim)
gradient_wrt_params = jacrev(model_output)(params, X[:10])
# grad_per_sample = vmap(grad(model_output), in_dims=(None, 0))(params, X)
predictions = model_output(params, X[:10])
product = (-(y[:10]-predictions)).T @ gradient_wrt_params['linear.0.weight'].view(10, -1)
product.view(100, -1)

In [ ]:
# Compute Jacobian of output w.r.t. weights
# x = torch.randn(10, input_dim)
# gradient_wrt_params = jacrev(model_output)(params, X[:10])
with torch.no_grad():
    # vector-Jacobian product, returns function (model_output) applied to primals (params, X) 
    # and a function that computes the vector-Jacobian product
    predictions, vjp_func = vjp(model_output, params, X[:10])
    # vjp with residuals, then select the derivative w.r.t. params
    vjp_result =  vjp_func(-(y[:10] - predictions))[0] 
print(vjp_result['linear.0.weight'])

## Dropout regularization in MNIST

Train a model without dropout

In [13]:
noiseless_model = NoisyMLP(dropout_rate=0.0, out_features=1)
wandb_logger = WandbLogger(project="inductive-bias", name="noiseless-mlp", log_model=True)
mnist = MNISTDataModule(batch_size=32)
trainer = Trainer(max_epochs=25,
                  logger=wandb_logger, 
                #   callbacks=[EarlyStopping(monitor="val/loss")]
                  )
trainer.fit(noiseless_model, mnist) 
wandb.finish()

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jrudoler/.cache/pypoetry/virtualenvs/inductive ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | loss_func | BCEWithLogitsLoss | 0      | train
1 | layers    | Sequential        | 150 K  | train
--------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=25` reached.


epoch,▁▁▁▁▁▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇████
train/loss,█▄▂▁▆▁▁▁▁▂▂▁▁▁▁▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
val/loss,▆▃▂▁▁▂▁▂▂▂▄▃▃▂▄▄▂▃▃▃█▄▄▅▃
epoch,24
train/loss,0.0
trainer/global_step,46874
val/loss,0.03689


Train a model with dropout, which we suspect is equivalent to some L2 regularization

In [ ]:
noisy_model = NoisyMLP(dropout_rate=0.3, out_features=1)  # Initialize the model with dropout
wandb_logger = WandbLogger(project="inductive-bias", name="noisy-mlp", log_model=True)
mnist = MNISTDataModule(batch_size=32)
trainer = Trainer(max_epochs=25,
                  logger=wandb_logger, 
                  callbacks=[EarlyStopping(monitor="val/loss")])
trainer.fit(noisy_model, mnist) 
wandb.finish()

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jrudoler/.cache/pypoetry/virtualenvs/inductive ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | loss_func | BCEWithLogitsLoss | 0      | train
1 | layers    | Sequential        | 150 K  | train
--------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇███
train/loss,▃▁▄▂▂▂▁▁▁▃▁▁▂▅▁▃▁▁▃▁▁▁▃▁▇▂▂▁▁▁▁▁▁▆█▄▁▁▁▅
trainer/global_step,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇██
val/loss,█▂▂▂▁▂▁▁▁▂
epoch,9
train/loss,0.00022
trainer/global_step,18749
val/loss,0.03147


In [14]:
noisy_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-sxjn7tc3:latest')
noiseless_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-n7awb9ov:latest')
no_stopping_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-yv15t2cq:v0')
mnist = MNISTDataModule(batch_size=32)

wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


In [ ]:
from core.bias import RidgeBias, BiasWithCrossEntropy
module = BiasWithCrossEntropy(
        predictive_model=noisy_model,
        bias_model=RidgeBias(),
        loss_fn=nn.functional.mse_loss,
        optimizer_cls=torch.optim.Adam,
        lr=1e-3,
    )

# Train using the Trainer interface.
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-noisy"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,▇▆▁▃▄▅▃▃▁▄█▄▃▄▄▃▄▆▇▆▃▄▅▅▆▅▄▃▄▁▁▂▅▇▇▃▃▃▄▁
epoch,▁▁▁▁▁▁▁▁▂▂▂▂▂▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▇▇▇███████
train/loss,▂█▂▂▂▁▂▃▂▂▂▂▂▃▂▁▁▁▂▂▂▄▁▃▃▂▂▂▁▃▂▁▁▂▁▂▂▂▁▂
trainer/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
bias/beta,0.50235
epoch,5
train/loss,302.12073
trainer/global_step,11249


In [ ]:
module = BiasWithCrossEntropy(
        predictive_model=noiseless_model,
        bias_model=RidgeBias(),
        loss_fn=nn.functional.mse_loss,
        optimizer_cls=torch.optim.Adam,
        lr=1e-3,
    )

# Train using the Trainer interface.
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-noiseless"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


bias/beta,█▆▄▃▃▂▂▁▁▁▂▁▁▂▁▁▁▁▁▁▁▂▁▂▁▁▁▁▁▁▁▂▂▁▂▁▂▁▁▁
epoch,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆███████████
train/loss,█▂▂▂▂▃▃▂▄▄▂▃▂▂▁▂▃▁▁▁▂█▃▂▂▂▂▂▄▂▂▄▃▄▂▂▃▂▃▂
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
bias/beta,0.3799
epoch,3
train/loss,25.0396
trainer/global_step,7499


In [15]:
module = BiasWithCrossEntropy(
    predictive_model=no_stopping_model,
    bias_model=RidgeBias(),
    loss_fn=nn.functional.mse_loss,
    optimizer_cls=torch.optim.Adam,
    lr=1e-3,
)
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-no-stopping"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,▁▂▃▅▅▆▆▇▇▇▇▇▆████▇▆▇███▇▇███▇▆▇████▇█▇█▆
epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆████████
train/loss,▁█▁▁▁▃▂▃▃▂▂▂▁▂▁▄▁▄▂▁▂▂▅▃▂▂▃▁▂▂▂▁▁▁▂▂▄▁▁▄
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
bias/beta,1.27503
epoch,3
train/loss,6461.88672
trainer/global_step,7499


In [4]:

def predictive_loss_grad(
        predictions: torch.Tensor, targets: torch.Tensor, loss_fn
    ) -> torch.Tensor:
        # Compute per-sample gradient of the loss function with respect to the model params
        loss = loss_fn(predictions, targets)
        # Compute the gradient of the loss with respect to the model predictions (dL/dy_hat)
        # because we're using chain rule.
        per_sample_grad = torch.autograd.grad(
            loss, predictions, retain_graph=True, create_graph=True
        )[0]
        return per_sample_grad

# unit tests

# Test the predictive_loss_grad function on mse loss
def test_predictive_loss_grad_mse():
    predictions = torch.tensor([[1.0], [2.0], [3.0]], requires_grad=True)
    targets = torch.tensor([[1.5], [2.5], [3.5]])
    loss_fn = nn.MSELoss(reduction="sum")
    
    # Compute the gradient
    grad = predictive_loss_grad(predictions, targets, loss_fn)
    
    # Expected gradient: 2 * (predictions - targets)
    expected_grad = 2 * (predictions - targets)
    
    assert torch.allclose(grad, expected_grad), f"Expected {expected_grad}, but got {grad}"

test_predictive_loss_grad_mse()